# Rigorous DLM relation runner
This notebook is a thin launcher for the repository CLI. Store `HF_TOKEN` in Colab Secrets, select one model environment, smoke-test it, and persist the complete run directory.

In [ ]:
!nvidia-smi
import os
import platform
import subprocess

print(platform.platform())
print(subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())

In [ ]:
MODEL = 'configs/models/dream_7b.yaml'
DATASET = 'configs/datasets/ewt.yaml'
EXPERIMENT = 'configs/experiments/head_search.yaml'
RUN_ID = 'replace-with-stable-id'
REQUIREMENTS = 'requirements/dream.txt'

In [ ]:
from google.colab import userdata

os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
!pip install -q -e .
!pip install -q -r {REQUIREMENTS}

In [ ]:
!python -m dlmrel.cli prepare --dataset {DATASET}
!python -m dlmrel.cli smoke-test --model {MODEL} --dataset {DATASET} --dry-run
RUN = ['python', '-m', 'dlmrel.cli', 'run', '--model', MODEL, '--dataset', DATASET]
RUN += ['--experiment', EXPERIMENT, '--run-id', RUN_ID]
subprocess.run(RUN + ['--dry-run'], check=True)

In [ ]:
# Run the real smoke test before the expensive experiment. Add --resume only after interruption.
!python -m dlmrel.cli smoke-test --model {MODEL} --dataset {DATASET}
subprocess.run(RUN, check=True)